In [26]:
import numpy as np
import csv

In [ ]:
# ===========================================
# 1. Leitura do CSV
# ===========================================

def load_data(filename):
    X = []
    y = []

    with open(filename, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)

        for row in reader:
            # Target
            survived = row['survived']
            if survived == '':
                continue

            y.append(int(survived))

            # Features selecionadas
            # Vamos usar:
            # pclass, sex, age, sibsp, parch, fare

            pclass = float(row['pclass']) if row['pclass'] != "" else 0

            # Convertendo sexo
            sex = 1 if row['sex'] == 'female' else 0

            age = float(row['age']) if row['age'] != "" else -1

            sibsp = float(row['sibsp']) if row['sibsp'] != "" else 0
            parch = float(row['parch']) if row['parch'] != "" else 0
            fare = float(row['fare']) if row['fare'] != "" else 0

            X.append([pclass, sex, age, sibsp, parch, fare])

    return np.array(X), np.array(y).reshape(-1, 1) #não sei quantas linhas tem, descubra sozinho". O 1 diz: "mas garanta que tenha apenas uma coluna"

    #X: Tabela larga (Matriz 2D).
    #y: Torre alta e fina (Matriz 2D, graças ao .reshape(-1, 1)).
    #Resultado: As duas são "caixas" (2D), o que torna a matemática entre elas muito mais rígida e segura.


In [ ]:
# ================================
# 2. Tratamento de Dados
# ================================

def fill_missing_age(X):
    ages = X[:, 2]
    mean_age = np.mean(ages[ages != -1])

    #A Posição: Ela foca na Coluna 2 (X[:, 2]).
#A Lógica: Ela calcula a média de todas as idades que são válidas (diferentes de -1) e substitui os "buracos" por essa média.
#Por que fazer isso? O computador não sabe fazer contas com "vazio" ou "-1" 
#(que puxaria a média de idade para baixo de forma irreal). Preencher com a média é uma técnica clássica para manter o passageiro no modelo sem inventar um dado extremo.

    X[:, 2] = np.where(ages == -1, mean_age, ages)
    return X

# ----------------------------------------------------
def normalize(X):
    mean = np.mean(X, axis=0) # pega média de cada coluna
    std = np.std(X, axis=0) # pega o desvio padrão 
    return (X - mean) / (std + 1e-8) #subtrai a media, depois divide pelo desvio padrão - [1e-8 pra garantir que não vai dividir por 0]

#O Problema: No Titanic, a coluna Fare (Preço da passagem) pode chegar a 500, enquanto a coluna SibSp (irmãos a bordo) costuma ser 0, 1 ou 2.O 
#Conflito: Se você não normalizar, o modelo vai achar que o Fare é 500 vezes mais importante que o resto só porque o número é maior.
#A Solução: A função subtrai a média e divide pelo desvio padrão. Isso faz com que todas as colunas fiquem na mesma "língua" (geralmente entre -3 e 3).
#O detalhe técnico (axis=0): Isso garante que a média seja calculada por coluna (verticalmente). 
#Você não quer misturar a média de idade com a média de preço!
# O 1e-8: É um "número de segurança" minúsculo para evitar que o Python tente dividir por zero se uma coluna tiver todos os valores iguais.

def add_bias(X):
    ones = np.ones((X.shape[0], 1))
    return np.hstack((ones, X)) 
    #Esta função faz exatamente o que fizemos no exercício das casas, mas usando uma técnica de "empilhamento".
    #np.ones((X.shape[0], 1)): Cria uma coluna de 1s com a mesma altura da sua matriz X.
    #np.hstack(Horizontal Stack): Pega essa coluna de 1s e "cola" na frente da sua matriz de características.
    

 

In [ ]:
# =============================
# 3. Funções do Modelo
# =============================

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

#Esta função é o que transforma qualquer número (positivo ou negativo, grande ou pequeno) em um valor entre 0 e 1.
#Posição: Ela recebe o resultado de X @ beta.
#Visualmente: Imagine que a conta das matrizes dá um número como 5.0. A Sigmoide "espreme" esse 5 e ele vira 0.99 (99% de chance de sobrevivência). 
#Se desse -5.0, viraria 0.01 (1% de chance).

def compute_cost(X, y, beta):
    m = len(y)
    h = sigmoid(X @ beta)
    epsilon = 1e-8

    cost = -(1/m) * np.sum(
        y * np.log(h + epsilon) + (1 - y) * np.log(1 - h + epsilon)
    )

    return cost

#Esta função mede o quão "ruim" o seu modelo está.
#h = sigmoid(X @ beta): O h é uma coluna de probabilidades (Ex: 0.8, 0.2, 0.9).
#y: É a coluna real de quem sobreviveu (Ex: 1, 0, 1).
#A Posição: O Python compara a coluna h com a coluna y. Se o modelo disse 0.8 (80%) e a pessoa realmente sobreviveu (1), o erro é pequeno. 
#Se ele disse 0.8 e a pessoa morreu (0), o erro (custo) aumenta muito.
#epsilon: Assim como na normalização, serve para evitar que o Python tente calcular log(0), o que daria erro matemático.


def gradient_descent(X, y, beta, lr, epochs):
    m = len(y)

    for i in range(epochs):
        h = sigmoid(X @ beta)
        gradient = (1/m) * (X.T @ (h - y))

        beta = beta - lr * gradient

        # if i % 100 == 0:
        #     print(f"Epoch {i} - Cost: {compute_cost(X, y, beta):.4f}")

    return beta

   # Este é o loop onde o aprendizado acontece. 
   # Vamos olhar para as posições das matrizes dentro do loop:A "Diferença" (h - y)
   #Posição: Você tem a coluna de previsões (h) e a coluna de verdades (y).Ao fazer h - y, você gera uma nova coluna chamada "Erro".
   #Se o erro for positivo, o modelo foi otimista demais; se for negativo, foi pessimista.
   #O Cálculo do Gradiente (X.T @ (h - y))
   #X.T (Transposta de X): Está "deitada" (7 linhas x N passageiros).(h - y): Está "em pé" (N passageiros x 1 coluna).
   #Resultado: Uma coluna de 7 números. Cada número diz para o modelo: "Para a característica X (como 'sexo' ou 'idade'), você deve aumentar ou diminuir o peso beta nesse tanto".
   #A Atualização dos Pesos (beta = beta - lr * gradient)lr 
   #(Learning Rate/Taxa de Aprendizado): É o tamanho do passo.O modelo pega os pesos atuais e subtrai o erro multiplicado pelo lr.
#Posição Final: O vetor beta (de 7 linhas) vai sendo "lapidado" a cada época (rodada) até que o erro (cost) pare de cair.
    

In [ ]:
# =====================
# 4. Treinamento
# =====================

def train(X, y):
    X = fill_missing_age(X)
    X = normalize(X)
    X = add_bias(X)

    beta = np.zeros((X.shape[1], 1))

    beta = gradient_descent(X, y, beta, lr=0.01, epochs=2000)

    return beta, X

 # Diferente da Regressão Linear, onde o beta é o resultado de uma conta direta, aqui você o cria "vazio".
 #X.shape[1]: O Python olha para a sua matriz X (que agora tem 7 colunas devido ao add_bias) e diz: "Preciso de 7 pesos".
 # A Posição: Ele cria uma torre vertical (Matriz de 7 linhas e 1 coluna).

 #O Fluxo de Dados na Função train
 #Essa função funciona como uma linha de montagem. 
 #Veja a ordem das tabelas:Limpeza: fill_missing_age tapa os buracos na matriz X
 # Ajuste: normalize coloca todos os números na mesma escala.
 #Expansão: add_bias coloca a coluna de 1s na frente, mudando o formato de X de [N, 6] para [N, 7].
 # Inicialização: O beta é criado com zeros para combinar com essas 7 colunas.
 #Refinamento: O gradient_descent recebe esse beta zerado e os dados reais, e começa a "moldar" os valores até que eles consigam prever quem sobreviveu.

In [ ]:
# =====================
# 5. Predição
# =====================

def predict(X, beta):
    probs = sigmoid(X @ beta)
    return (probs >= 0.5).astype(int) #Converte resultado da comparação [True ou False] em 1 OU 0


#Na função predict, você usa os pesos betas que o modelo aprendeu no treino para classificar novos passageiros.
#probs = sigmoid(X @ beta): Aqui, o resultado é uma coluna de probabilidades (ex: 0.72, 0.45, 0.10).
#(probs >= 0.5): Este é o "juiz". Se a probabilidade for 0.51, o modelo diz "Sobreviveu". 
#Se for 0.49, diz "Morreu".
#astype(int): Transforma os valores de True/False em 1 (Sobreviveu) ou 0 (Morreu).
#A Posição das Tabelas:Nesse momento, você tem duas colunas verticais (vetores) idênticas em tamanho:y_true: O que realmente aconteceu no Titanic.
#y_pred: O que o seu modelo calculou.

def confusion_matrix(y_true, y_pred):
    # Achatar os arrays para garantir que tenham apenas 1 dimensão
    y_t = y_true.flatten() #TUDO QUE É VERDADEIRO NO DATASET
    y_p = y_pred.flatten() #RESULTADO DA FUNÇÃO PREDICT E DE TODA REGRESSÃO LOGISTICA

    # O operador '&' faz a comparação bit a bit (element-wise) nos arrays do numpy
    # Faz AND lógico, elemento por elemento de cada array.
    # Por exemplo:
    #   y_t = np.array([1, 0, 1, 1])
    #   cond1 = (y_t == 1)
    #   Resultado: [True, False, True, True]
    VP = np.sum((y_t == 1) & (y_p == 1))
    VN = np.sum((y_t == 0) & (y_p == 0))
    FP = np.sum((y_t == 0) & (y_p == 1))
    FN = np.sum((y_t == 1) & (y_p == 0))

    return VP, VN, FP, FN

def calculate_metrics(VP, VN, FP, FN):
    acc = (VP + VN) / (VP + VN + FP + FN)
    prec = VP / ( VP + FP )
    rec = VP / ( VP + FN )
    spec = VN / (VN + FP)
    f1 = 2 * (prec * rec) / (prec + rec)

    return acc, prec, rec, spec, f1

In [ ]:
# A função confusion_matrix compara essas duas colunas.
#Imagine uma tabela 2x2:Como o código encontra os valores (Lógica &):
#O código usa o operador & para verificar onde as duas colunas concordam ou discordam:
#VP (Verdadeiro Positivo): O passageiro sobreviveu (y_t == 1) E o modelo acertou (y_p == 1).
#VN (Verdadeiro Negativo): O passageiro morreu (y_t == 0) E o modelo acertou (y_p == 0).
#FP (Falso Positivo): O passageiro morreu (y_t == 0), MAS o modelo disse que ele sobreviveria (y_p == 1). 
#Também chamado de Erro Tipo I.
#FN (Falso Negativo): O passageiro sobreviveu (y_t == 1), MAS o modelo disse que ele morreria (y_p == 0). Também chamado de Erro Tipo II.
#Métricas: O que elas dizem?
#A função calculate_metrics transforma esses quatro números em porcentagens compreensíveis:
#Acurácia (Acc) No geral, quanto o modelo acertou
#Precisão (Prec) De todos que o modelo disse que sobreviveram, quantos realmente sobreviveram?
#Recall (Rec) De todos que realmente sobreviveram, quantos o modelo conseguiu achar?
#Especificidade De todos que morreram, quantos o modelo previu corretamente?
#F1-Score O equilíbrio entre Precisão e Recall (ótimo para dados desbalanceados). Média Harmônica

In [32]:
# ===========================
# 6. Split treino / teste
# ===========================

def train_test_split(X, y, test_size=0.2):
    np.random.seed(42)

    indices = np.arange(len(X))
    np.random.shuffle(indices)

    split = int(len(X) * (1 - test_size))

    train_idx = indices[:split]
    test_idx = indices[split:]

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

In [ ]:
#Embaralhamento (shuffle): Primeiro, ela embaralha os índices. Isso é vital! Imagine que o arquivo do Titanic estivesse ordenado por classe (primeira classe no topo). 
#Se você pegasse os primeiros 80%, você só treinaria com gente rica. 
#O shuffle garante que o treino tenha uma mistura de tudo.
#O Corte (split): Se você tem 1000 linhas e o test_size é 0.2 (20%), o código define que 800 linhas são para treino e 200 para teste.
#As novas dimensões das tabelas:
#X_train / y_train: O bloco maior (80%). Você entrega esses para a função train(X, y).
#X_test / y_test: O bloco menor (20%). Você guarda esses na gaveta.

In [33]:
# =====================
# 7. Execução
# =====================

X, y = load_data("titanic.csv")

X_train, X_test, y_train, y_test = train_test_split(X, y)

beta, X_train_processed = train(X_train, y_train)

# Processar teste com MESMAS transformações
X_test = fill_missing_age(X_test)
X_test = normalize(X_test)
X_test = add_bias(X_test)

# Vetor de Predições
y_pred = predict(X_test, beta)

# Métricas da Matriz de Confusão
VP, VN, FP, FN = confusion_matrix(y_test, y_pred)

print("\n" + "="*30)
print(' Matriz de Confusão ')
print('='*30)
print(f'Verdadeiros Positivos (VP): {VP}')
print(f'Verdadeiros Negativos (VN): {VN}')
print(f'Falsos Positivos (FP): {FP}')
print(f'Falsos Negativos (FN): {FN}')

# Calcular e exibir as métricas
acc, prec, rec, spec, f1 = calculate_metrics(VP, VN, FP, FN)

print('\n' + '='*30)
print(' Métricas de Avaliação')
print('='*30)
print(f'Acurácia:   {acc:.4f} ({(acc*100):.1f}%)')
print(f'Precisão:   {prec:.4f} ({(prec*100):.1f}%)')
print(f'Revocação:   {rec:.4f} ({(rec*100):.1f}%)')
print(f'Especificidade:   {spec:.4f} ({(spec*100):.1f}%)')
print(f'F1-Score:   {f1:.4f} ({(f1*100):.1f}%)')


 Matriz de Confusão 
Verdadeiros Positivos (VP): 68
Verdadeiros Negativos (VN): 123
Falsos Positivos (FP): 32
Falsos Negativos (FN): 39

 Métricas de Avaliação
Acurácia:   0.7290 (72.9%)
Precisão:   0.6800 (68.0%)
Revocação:   0.6355 (63.6%)
Especificidade:   0.7935 (79.4%)
F1-Score:   0.6570 (65.7%)


In [ ]:
load_data: Pega tudo.

train_test_split: Divide em dois grupos (Treino e Teste).

train(X_train, y_train): O modelo estuda apenas o grupo de treino e gera o beta.

predict(X_test, beta): O modelo faz a prova usando os dados de teste que ele nunca viu. Gera o y_pred.

confusion_matrix(y_test, y_pred): Você corrige a prova comparando a resposta real do teste com o que o modelo chutou.


A Regressão Logística como algoritmo acontece no momento em que você chama o train e o predict. Mas para que ela seja honesta e científica, você precisa desse Passo 6 antes para garantir que não está "roubando" ao testar o modelo com dados que ele já conhecia.